In [1]:
%load_ext autoreload
%autoreload 2

### Imports and constants

In [ ]:
import os
import re
from pathlib import Path

import h5py
import pandas as pd

from roar import ROOT_DIR
from roar.preprocessing import extract_features_from_h5_file

In [ ]:
MIC_CHANNELS = {
    "TrailK1": "Ch_1_labV12",
    "TrailK2": "Ch_2_labV12",
    "LeadK1": "Ch_3_labV12",
    "LeadK2": "Ch_4_labV12",
    "NAWSSound": "NAWSSound",
    "mic_iso": "mic_iso",
}

tracks = ["track150", "track211"]
cars = ["01_ID.4", "02_Q8 e-tron", "03_Taycan", "04_E-Golf"]
tyres = ["tyre1", "tyre3", "tyre6", "tyre10", "tyre12", "tyre13"]

experiment_mapping = {
    "track150": {
        "01_ID.4": ["tyre1"],
        "02_Q8 e-tron": ["tyre6", "tyre12"],
        "03_Taycan": ["tyre10"],
        "04_E-Golf": ["tyre13"],
    },
    "track211": {
        "01_ID.4": ["tyre1", "tyre3"],
        "02_Q8 e-tron": ["tyre6", "tyre12"],
        "03_Taycan": ["tyre10"],
        "04_E-Golf": ["tyre13"],
    },
}

### Renaming the ikaISO files (track 259)

In [ ]:
## Renaming helpers
def rename_h5_files(directory):
    pattern = re.compile(
        r"""
        ^(?P<prefix>b\d{2}_)?                # optional b##
        (?P<vehicle>ID4|Q8\ e-tron|Taycan|E-Golf)_
        (?P<tyre>[\w-]+)_
        (?P<test>ikaISO|ikaTP|ikaST)_
        (?P<amplitude>[^_]+)_
        (?P<speed>vr\d{2,3})_
        (?P<rest>.+\.h5)$
        """,
        re.VERBOSE,
    )

    for path in Path(directory).glob("*.h5"):
        match = pattern.match(path.name)
        if not match:
            continue

        prefix = (match.group("prefix") or "").rstrip("_")
        parts = [
            match.group("test"),
            match.group("vehicle"),
            match.group("tyre"),
            match.group("amplitude"),
            match.group("speed"),
        ]
        if prefix:
            parts.append(prefix)

        parts.append(match.group("rest"))
        new_name = "_".join(parts)

        if new_name != path.name:
            new_path = path.with_name(new_name)
            print(f"Renaming {path.name} -> {new_name}")
            path.rename(new_path)


def retag_h5_files(directory):
    test_map = {"track150": "track211"}
    vehicle_map = {"ID.4": "ID.4"}
    tyre_map = {"tyre3": "tyre3"}
    pattern = re.compile(r"^(?P<test>[^_]+)_(?P<vehicle>[^_]+)_(?P<tyre>[^_]+)_(?P<rest>.+)$")

    for path in Path(directory).glob("*.h5"):
        match = pattern.match(path.name)
        if not match:
            continue

        test = test_map.get(match.group("test"), match.group("test"))
        vehicle = vehicle_map.get(match.group("vehicle"), match.group("vehicle"))
        tyre = tyre_map.get(match.group("tyre"), match.group("tyre"))

        new_name = f"{test}_{vehicle}_{tyre}_{match.group('rest')}"
        if new_name != path.name:
            print(f"Renaming {path.name} -> {new_name}")
            path.rename(path.with_name(new_name))

In [ ]:
target_dir = ROOT_DIR / "data_cleaned" / "259" / "01 VW ID4" / "02 UniRoyal RainSport5 - Tyre3"
rename_h5_files(target_dir)

Renaming ID4_RainSport5_ikaISO_2pt6_vr100_2025-07-11_10-32-06.h5 -> ikaISO_ID4_RainSport5_2pt6_vr100_2025-07-11_10-32-06.h5
Renaming ID4_RainSport5_ikaISO_2pt6_vr100_2025-07-11_10-32-58.h5 -> ikaISO_ID4_RainSport5_2pt6_vr100_2025-07-11_10-32-58.h5
Renaming ID4_RainSport5_ikaISO_2pt6_vr100_2025-07-11_10-33-49.h5 -> ikaISO_ID4_RainSport5_2pt6_vr100_2025-07-11_10-33-49.h5
Renaming ID4_RainSport5_ikaISO_3pt1_vr100_2025-07-11_09-48-47.h5 -> ikaISO_ID4_RainSport5_3pt1_vr100_2025-07-11_09-48-47.h5
Renaming ID4_RainSport5_ikaISO_3pt1_vr100_2025-07-11_09-49-51.h5 -> ikaISO_ID4_RainSport5_3pt1_vr100_2025-07-11_09-49-51.h5
Renaming ID4_RainSport5_ikaISO_3pt1_vr100_2025-07-11_09-50-40.h5 -> ikaISO_ID4_RainSport5_3pt1_vr100_2025-07-11_09-50-40.h5
Renaming ID4_RainSport5_ikaISO_3pt1_vr100_2025-07-11_09-51-30.h5 -> ikaISO_ID4_RainSport5_3pt1_vr100_2025-07-11_09-51-30.h5
Renaming ID4_RainSport5_ikaISO_3pt1_vr100_2025-07-11_09-52-21.h5 -> ikaISO_ID4_RainSport5_3pt1_vr100_2025-07-11_09-52-21.h5
Renaming

In [ ]:
target_dir = ROOT_DIR / "data_cleaned" / "track211" / "01_ID.4" / "tyre3"

retag_h5_files(target_dir)

Renaming track150_ID.4_tyre3_2pt6_vr100_2025-07-11_10-32-06.h5 -> track211_ID.4_tyre3_2pt6_vr100_2025-07-11_10-32-06.h5
Renaming track150_ID.4_tyre3_2pt6_vr100_2025-07-11_10-32-58.h5 -> track211_ID.4_tyre3_2pt6_vr100_2025-07-11_10-32-58.h5
Renaming track150_ID.4_tyre3_2pt6_vr100_2025-07-11_10-33-49.h5 -> track211_ID.4_tyre3_2pt6_vr100_2025-07-11_10-33-49.h5
Renaming track150_ID.4_tyre3_2pt6_vr45_2025-07-11_10-24-28.h5 -> track211_ID.4_tyre3_2pt6_vr45_2025-07-11_10-24-28.h5
Renaming track150_ID.4_tyre3_2pt6_vr45_2025-07-11_10-25-31.h5 -> track211_ID.4_tyre3_2pt6_vr45_2025-07-11_10-25-31.h5
Renaming track150_ID.4_tyre3_2pt6_vr45_2025-07-11_10-26-33.h5 -> track211_ID.4_tyre3_2pt6_vr45_2025-07-11_10-26-33.h5
Renaming track150_ID.4_tyre3_2pt6_vr50_2025-07-11_10-35-11.h5 -> track211_ID.4_tyre3_2pt6_vr50_2025-07-11_10-35-11.h5
Renaming track150_ID.4_tyre3_2pt6_vr50_2025-07-11_10-36-17.h5 -> track211_ID.4_tyre3_2pt6_vr50_2025-07-11_10-36-17.h5
Renaming track150_ID.4_tyre3_2pt6_vr50_b35_2025-07

### NAMING INCONSISTENCY PROBLEMS

In [ ]:
# File Naming suggests road type 150, metadata says 211

etron_testfilepath = (
    ROOT_DIR
    / "data_cleaned"
    / "data_original"
    / "02 Audi Q8"
    / "ikastandard"
    / "Tyre12"
    / "01_2025-08-15"
    / "track150_Q8 e-tron_tyre12_meas0_2p5_1_2025-08-15_13-06-24.h5"
)
etron_cleanedfilepath = (
    ROOT_DIR
    / "data_cleaned"
    / "track150"
    / "02_Q8 e-tron"
    / "tyre12"
    / "track150_Q8 e-tron_tyre12_meas0_2p5_1_2025-08-15_13-06-24.h5"
)
print("Original ETron file metadata:\n")
with h5py.File(etron_testfilepath, "r") as h5file:
    for meta_key in h5file.attrs.keys():
        print(f"{meta_key} = {h5file.attrs[meta_key]}")
print()
print("Cleaned ETron file metadata:\n")
with h5py.File(etron_cleanedfilepath, "r") as h5file:
    for meta_key in h5file.attrs.keys():
        print(f"{meta_key} = {h5file.attrs[meta_key]}")

Original ETron file metadata:

CreationDate = 2025-08-15_13-06-24
MeasCount = [1.]
MeasName = Select Measurement
MeasTypeID = [0.]
Pressure = [2.5]
TrackID = [211.]
TrackName = ika Teststrecke
TyreID = [12.]
TyreName = R12 - Hankook Ventus S1 evo3 ev 255/50R20 109H
carName = Q8 e-tron

Cleaned ETron file metadata:

CreationDate = 2025-08-15_13-06-24
MeasCount = [1.]
MeasName = Select Measurement
MeasTypeID = [0.]
Pressure = [2.5]
TrackID = [211.]
TrackName = ika Teststrecke
TyreID = [12.]
TyreName = R12 - Hankook Ventus S1 evo3 ev 255/50R20 109H
carName = Q8 e-tron


In [ ]:
# File Naming suggests road type 211, metadata says 150

ikaISO_testfilepath = (
    ROOT_DIR
    / "data_cleaned"
    / "data_original"
    / "01 VW ID4"
    / "01 ika ISO-Akustikstrecke track211"
    / "02 UniRoyal RainSport5 - Tyre3"
    / "b35_ID4_RainSport5_ikaISO_2pt6_vr50_2025-07-11_10-37-31.h5"
)
ikaISO_cleanedfilepath = (
    ROOT_DIR
    / "data_cleaned"
    / "track259"
    / "01_ID.4"
    / "tyre3"
    / "track259_ID.4_tyre3_2pt6_vr45_2025-07-11_10-24-28.h5"
)
print("Original IkaISO file metadata:\n")
with h5py.File(ikaISO_testfilepath, "r") as h5file:
    for meta_key in h5file.attrs.keys():
        print(f"{meta_key} = {h5file.attrs[meta_key]}")
print()
print("Cleaned IkaISO file metadata:\n")
with h5py.File(ikaISO_cleanedfilepath, "r") as h5file:
    for meta_key in h5file.attrs.keys():
        print(f"{meta_key} = {h5file.attrs[meta_key]}")

Original IkaISO file metadata:

CreationDate = 2025-07-11_10-37-31
MeasCount = [1.]
MeasName = _Meas
MeasTypeID = [0.]
Pressure = ['2pt6']
TrackID = [150.]
TrackName = ['Road']
TyreID = ['']
TyreName = _UniRoyal RainSport5
carName = ['Vehicle']

Cleaned IkaISO file metadata:

CreationDate = 2025-07-11_10-24-28
MeasCount = [1.]
MeasName = _Meas
MeasTypeID = [0.]
Pressure = ['2pt6']
TrackID = [150.]
TrackName = ['Road']
TyreID = ['']
TyreName = _UniRoyal RainSport5
carName = ['Vehicle']


In [ ]:
# List all test files and their metadata reveals mismatches

data_path = ROOT_DIR / "data_cleaned"
meta_keys = ["TrackID", "carName", "TyreID"]

test_paths = []

for track in tracks:
    for car in cars:
        for tyre in experiment_mapping.get(track, {}).get(car, []):
            if not tyre:
                continue
            test_path = data_path / track / car / tyre
            test_paths.append(test_path)

for test_path in test_paths:
    print(f"[{test_path.relative_to(data_path)}]\n")
    for testfilepath in os.listdir(test_path):
        if testfilepath.endswith(".h5"):
            full_testfilepath = test_path / testfilepath
            with h5py.File(full_testfilepath, "r") as h5file:
                meta_values = []
                for meta_key in meta_keys:
                    meta_values.append(str(h5file.attrs.get(meta_key, "N/A")))
                print(", ".join(meta_values))
    print()

[track150\01_ID.4\tyre1]

[150.], ID.4, [1.]
[150.], ID.4, [1.]
[150.], ID.4, [1.]
[150.], ID.4, [1.]
[150.], ID.4, [1.]
[150.], ID.4, [1.]
[150.], ID.4, [1.]
[150.], ID.4, [1.]
[150.], ID.4, [1.]

[track150\01_ID.4\tyre3]

[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], ['']
[150.], ['Vehicle'], [

### Single scalar audio feature extraction for baseline models and name parser

In [ ]:
from roar import (
    DATA_DIR,
    MIC_CHANNELS,  # noqa: F811
)

# Test feature extraction
features = extract_features_from_h5_file(
    DATA_DIR
    / "track211/02_Q8 e-tron/tyre12/track211_Q8 e-tron_tyre12_meas1_2p5_1_2025-08-15_11-46-48.h5",
    MIC_CHANNELS,
)
for feature in features.keys():
    print(f"{feature}: {features[feature]}")
print(len(features))

Ch_1_labV12_rms: 1.9825720167974183
Ch_1_labV12_mean: -7.507651404992822e-05
Ch_1_labV12_std: 1.9825720153759108
Ch_1_labV12_max: 8.968762397766113
Ch_1_labV12_crest: 4.523801565468555
Ch_1_labV12_zcr: 0.008376712625777952
Ch_1_labV12_spec_centroid: 76.32497673989454
Ch_1_labV12_spec_rolloff: 445.31249999999926
Ch_1_labV12_spec_flatness: 3.621808391661969e-05
Ch_1_labV12_spec_bandwidth: 205.7811301947801
Ch_1_labV12_band_0: 0.26112378989874435
Ch_1_labV12_band_1: 0.007389678246942566
Ch_1_labV12_band_2: 0.009410983385702936
Ch_1_labV12_band_3: 0.0033581412924817682
Ch_1_labV12_band_4: 0.00013030426464310808
Ch_1_labV12_mfcc_1: -26.38875389099121
Ch_1_labV12_mfcc_2: 213.49420166015625
Ch_1_labV12_mfcc_3: 20.73758888244629
Ch_1_labV12_mfcc_4: -15.789551734924316
Ch_1_labV12_mfcc_5: 2.5844290256500244
Ch_1_labV12_mfcc_6: 1.983687400817871
Ch_1_labV12_mfcc_7: 7.090250492095947
Ch_1_labV12_mfcc_8: 9.382965087890625
Ch_1_labV12_mfcc_9: 6.579596042633057
Ch_1_labV12_mfcc_10: 5.388495922088623

### Build the initial dataset

In [ ]:
from roar.preprocessing import parse_filename

# Get the h5 file paths
h5_paths = list(DATA_DIR.rglob("*.h5"))
print(f"Found {len(h5_paths)} files")

rows = []
for path in h5_paths:
    feat_dict = extract_features_from_h5_file(path, channels=MIC_CHANNELS)
    meta = parse_filename(path)

    row = {
        **feat_dict,
        **meta,
    }
    rows.append(row)

df = pd.DataFrame(rows)

df.info(verbose=True, show_counts=True)

Found 204 files
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 203 columns):
 #    Column                      Non-Null Count  Dtype         
---   ------                      --------------  -----         
 0    Ch_1_labV12_rms             204 non-null    float64       
 1    Ch_1_labV12_mean            204 non-null    float64       
 2    Ch_1_labV12_std             204 non-null    float64       
 3    Ch_1_labV12_max             204 non-null    float64       
 4    Ch_1_labV12_crest           204 non-null    float64       
 5    Ch_1_labV12_zcr             204 non-null    float64       
 6    Ch_1_labV12_spec_centroid   204 non-null    float64       
 7    Ch_1_labV12_spec_rolloff    204 non-null    float64       
 8    Ch_1_labV12_spec_flatness   204 non-null    float64       
 9    Ch_1_labV12_spec_bandwidth  204 non-null    float64       
 10   Ch_1_labV12_band_0          204 non-null    float64       
 11   Ch_1_labV12_band_1         